# 📓 07 — Robot Tool & Agent IntegrationThe `Robot` class is a Strands `AgentTool` — it can be used by an AI agent to controlphysical robots with natural language, or called directly from Python code.

## Robot as AgentToolThe Robot class implements the Strands `AgentTool` interface with 4 actions:| Action | Type | Description ||--------|------|-------------|| `execute` | Blocking | Run until duration expires or task completes || `start` | Async | Start task in background, return immediately || `status` | Query | Get current task status (IDLE/RUNNING/COMPLETED/ERROR) || `stop` | Control | Interrupt a running task |

## Constructor```pythonfrom strands_robots import Robotrobot = Robot(    tool_name="orange_arm",        # unique name for this robot    robot="so101_follower",        # LeRobot config type or Robot instance    cameras={        "front": {"type": "opencv", "index_or_path": "/dev/video0", "fps": 30},        "wrist": {"type": "opencv", "index_or_path": "/dev/video2", "fps": 30},    },    port="/dev/ttyACM0",           # serial port for Feetech servos    data_config="so100_dualcam",   # GR00T data config    control_frequency=50.0,        # Hz (action_sleep_time = 1/freq)    action_horizon=8,              # actions per inference step)```

## Using with a Strands Agent```pythonfrom strands import Agentfrom strands_robots import Robot, gr00t_inference# Create robot + agentrobot = Robot(tool_name="my_arm", robot="so101_follower", ...)agent = Agent(tools=[robot, gr00t_inference])# Start GR00T inferenceagent.tool.gr00t_inference(action="start", checkpoint_path="...", port=8000)# Natural language controlagent("Use my_arm to pick up the red block using GR00T policy on port 8000")# The agent translates this to:# robot.execute(instruction="pick up the red block", policy_port=8000, ...)```

## Async Task Execution```python# Start task (returns immediately)agent("Start my_arm picking up the cube on port 8000")# Check progressagent("What's the status of my_arm?")# → "RUNNING: 45 steps, 2.3s elapsed"# Stop if neededagent("Stop my_arm")```

## Direct Python API```python# Blocking executionresult = robot._execute_task_sync(    instruction="wave arm",    policy_port=8000,    policy_host="localhost",    policy_provider="groot",    duration=10.0,)# Async startresult = robot.start_task("wave arm", policy_port=8000)status = robot.get_task_status()robot.stop_task()```

## Task State Machine```IDLE → CONNECTING → RUNNING → COMPLETED                       ↓                    STOPPED                       ↓                     ERROR```The `RobotTaskState` tracks: status, instruction, start_time, duration,step_count, error_message, and the background `Future`.

## Policy Integration Flow```python# Inside _execute_task_async():1. Connect robot hardware (LeRobot)2. Create policy via create_policy(provider, port=...)3. Initialize policy with robot state keys4. Control loop:   a. obs = robot.get_observation()          # cameras + joints   b. actions = policy.get_actions(obs, instruction)  # VLA inference   c. for action in actions[:action_horizon]:        robot.send_action(action)        sleep(1/control_frequency)           # 50Hz default5. Report completion```